In [ ]:
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time

def scrape_rextreme_full_process():
    options = uc.ChromeOptions()
    options.add_argument("--start-maximized")
    
    # 🌟 현재 크롬 버전인 145를 명시적으로 지정합니다.
    driver = uc.Chrome(options=options, version_main=145) 
    
    results = []

    try:
        # 1. 로그인 단계
        driver.get("https://www.the-b.co")
        print("\n" + "="*60)
        print("⚠️  [단계 1: 로그인] 열린 창에서 로그인을 완료한 후 엔터를 누르세요.")
        input("="*60 + "\n로그인 완료 후 Enter!")

        # 2. 리뷰 페이지 이동
        driver.get("https://www.the-b.co/reviews?brand=rextreme")
        time.sleep(5) 

        # 3. 기간 설정 (전체 기간)
        print("📅 기간 필터를 '전체 기간'으로 변경 중...")
        try:
            # 이미지(image_5b7441.png)에서 확인된 날짜 버튼 클릭
            # 클래스명 일부를 조합하여 정확하게 타겟팅합니다.
            date_btn_selector = "button.border-\\[\\#E5E5E5\\]"
            date_filter_btn = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, date_btn_selector))
            )
            driver.execute_script("arguments[0].click();", date_filter_btn)
            time.sleep(2)

            # '전체 기간' 옵션 클릭 (텍스트 기반 매칭)
            # 보통 드롭다운 내의 '전체' 혹은 '전체 기간' 텍스트를 찾습니다.
            all_period_opt = driver.find_element(By.XPATH, "//*[contains(text(), '전체')]")
            driver.execute_script("arguments[0].click();", all_period_opt)
            time.sleep(3) # 필터 적용 대기
            print("✅ 전체 기간 설정 완료")
        except Exception as e:
            print(f"⚠️ 기간 설정 자동화 실패: {e}")
            input("수동으로 '전체 기간' 설정 후 엔터를 눌러주세요...")

        # 4. 무한 스크롤 (4,739건 로드)
        print("🔄 무한 스크롤 시작... (중단하려면 컨트롤+C)")
        last_height = driver.execute_script("return document.body.scrollHeight")
        while True:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2.5)
            new_height = driver.execute_script("return document.body.scrollHeight")
            if new_height == last_height:
                break
            last_height = new_height

        # 5. 원문 버튼 클릭 (텍스트 확장)
        print("📄 모든 원문 펼치는 중...")
        original_btns = driver.find_elements(By.XPATH, "//button[contains(., '원문')]")
        for btn in original_btns:
            try:
                driver.execute_script("arguments[0].click();", btn)
            except: continue

        # 6. 데이터 파싱 및 저장
        print("🧐 데이터 추출 및 저장 중...")
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        cards = soup.select('div.p-3.lg\\:p-4.flex-1.flex.flex-col')
        
        for card in cards:
            author = card.select_one('span.truncate').text.strip() if card.select_one('span.truncate') else ""
            content = card.find('p').text.strip() if card.find('p') else ""
            date = card.find('span', class_=lambda c: c and 'text-[#999999]' in c).text.strip() if card.find('span', class_=lambda c: c and 'text-[#999999]' in c) else ""
            
            results.append({'작성자': author, '작성일': date, '내용': content})

        df = pd.DataFrame(results)
        df.to_csv('rextreme_reviews_all.csv', index=False, encoding='utf-8-sig')
        print(f"✅ 저장 완료! 총 {len(results)}건")

    finally:
        driver.quit()

if __name__ == "__main__":
    scrape_rextreme_full_process()